# TinyMWAMNet · Notebook 05 — Microcontroller-faithful analysis, activation overflow, statistics and XAI

1. **Runtime agreement.** Do LiteRT's reference kernels, TensorFlow Lite Micro (the microcontroller runtime) and XNNPACK give the same INT8 outputs? (All images for XNNPACK vs reference; 48 images per model for TFLM, whose Python runtime is slow.)
2. **Activation overflow.** For every INT8 image: the share of activations that hit the limits of their INT8 range, which were fixed on in-domain calibration images. Compared across datasets, between flipped and unflipped images, and between ReLU and ReLU6 students.
3. **Statistics.** Seed-level summaries, paired per-seed INT8 costs, McNemar tests with Holm correction and paired bootstrap intervals (median seed), prediction flips, and logit margins.
4. **XAI stability.** Occlusion maps of the FP32 and INT8 students, and of the teacher.

In [ ]:
#@title 1 · Install packages
import importlib.metadata as md, subprocess, sys, os
def _v(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None
tfv = _v("tensorflow")
pk = []
if tfv is None:                         # Colab normally ships TensorFlow; install if absent
    pk.append("tensorflow[and-cuda]==2.21.0"); tfv = "2.21.0"
mm = ".".join(tfv.split(".")[:2])
pk += [f"tf_keras~={mm}.0", "setuptools<81"] + ['tflite-micro==0.dev20251201172741', 'statsmodels', 'ai-edge-litert']
if not os.environ.get("TMN_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pk, check=True)
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # tf_keras (Keras 2) is required by TF-MOT for QAT
print("TensorFlow", tfv, "| installed:", pk)

TensorFlow 2.20.0 | installed: ['tf_keras~=2.20.0', 'setuptools<81', 'tflite-micro==0.dev20251201172741', 'statsmodels', 'ai-edge-litert']


In [ ]:
%%writefile /content/tmn_common.py
"""
TinyMWAMNet — shared utilities for all Colab notebooks.

"""
import os, sys, json, math, time, glob, hashlib, random, gc, re, subprocess, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np

VERSION = "30 Sep 2026"      # printed by cell 3 of every notebook

# ---------------------------------------------------------------------------
# 1. Configuration (pre-specified; nothing here is tuned on shifted data)
# ---------------------------------------------------------------------------
CFG = dict(
    # Kaggle dataset identifiers
    DIN_ID="alik05/forest-fire-dataset",                 # in-domain (paper 1)
    WILD_ID="elmadafri/the-wildfire-dataset",            # Shift-1 (anticipated)
    DFIRE_ID="sayedgamal99/smoke-fire-detection-yolo",   # Shift-2 (unanticipated, CC0)
    IMG=224, SEED=42, SEEDS=[42, 43, 44, 45, 46], VAL_FRAC=0.20,
    # teacher (identical to paper 1)
    TEACHER_COPIES=4, HUE=0.10, L2=0.01, DROPOUT=0.5, LR=1e-4, WD=1e-4,
    EPOCHS=100, BATCH=32, PATIENCE=15,
    # augmentation pools for students
    STD_COPIES=8,          # paper-1 augmentation; copies 1-4 are identical to paper 1
    CONF_COPIES=4,         # confounder-aware copies (CAD)
    # students
    KD_T=2.0, KD_ALPHA=0.5,
    ST_BATCH=64, ST_WARM_EPOCHS=3, ST_EPOCHS=25, ST_LR_WARM=1e-3, ST_LR=3e-4,
    ST_WD=1e-4, ST_DROPOUT=0.2, QAT_EPOCHS=5, QAT_LR=5e-5,
    QAT_WARMUP_STEPS=3000,  # forward passes that let tfmot's EMA activation ranges converge (see qat_range_warmup)
    CALIB_N=256,
    # leakage audit
    PHASH_MAX_DIST=6,
    # weight-only bit-width sweep
    WBITS=[8, 6, 5, 4, 3],
    TEST_MODE=bool(int(os.environ.get("TMN_TEST_MODE", "0"))),
)

MAIN_STUDENT = ("mnv3s", 1.0, 128)          # TinyMWAMNet-S
RELU6_STUDENT = ("mnv3s6", 1.0, 128)        # same network, every ReLU capped at 6 (causal test)
FAMILY = [("mnv1", 0.25, 96),                # TinyMWAMNet-XS (MLPerf Tiny VWW size)
          ("mnv3s", 1.0, 96), ("mnv3s", 1.0, 160), ("mnv2", 0.35, 128)]
REGIMES = {  # name: (use KD?, pool kind)
    "CE-std": (False, "std"), "CE-conf": (False, "conf"),
    "KD-std": (True, "std"), "CAD": (True, "conf"),   # CAD = KD + confounder aug
}
EVALSETS = ("din", "wild", "dfire")
EVAL_NAMES = {"din": "In-domain test", "wild": "Shift-1 (The Wildfire Dataset)",
              "dfire": "Shift-2 (D-Fire)"}

if CFG["TEST_MODE"]:
    CFG.update(SEEDS=[42, 43], EPOCHS=3, PATIENCE=2, STD_COPIES=2, CONF_COPIES=2,
               TEACHER_COPIES=1, ST_WARM_EPOCHS=1, ST_EPOCHS=1, QAT_EPOCHS=1,
               CALIB_N=8, WBITS=[8, 4], QAT_WARMUP_STEPS=20)
    FAMILY = [("mnv1", 0.25, 96)]

IMGEXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


# ---------------------------------------------------------------------------
# 2. Paths (Google Drive if mounted, else local)
# ---------------------------------------------------------------------------
def setup_paths():
    root = os.environ.get("TMN_ROOT")
    if not root:
        root = ("/content/drive/MyDrive/TinyMWAMNet"
                if os.path.isdir("/content/drive/MyDrive") else "/content/TinyMWAMNet")
    P = {k: os.path.join(root, k) for k in
         ["data", "features", "models", "results", "preds", "figures", "tflite", "logs"]}
    P["root"] = root
    P["tmp"] = os.environ.get("TMN_TMP", "/content/tmn_tmp")
    for v in P.values():
        os.makedirs(v, exist_ok=True)
    return P


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))


def load_json(path):
    with open(path) as f:
        return json.load(f)


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    try:
        import tf_keras as keras
        keras.utils.set_random_seed(seed)
    except Exception:
        pass


# ---------------------------------------------------------------------------
# 3. Dataset download and discovery
# ---------------------------------------------------------------------------
def kaggle_download(kid):
    """kagglehub download; TMN_FAKE_DATA=<dir> redirects to local copies (testing)."""
    fake = os.environ.get("TMN_FAKE_DATA")
    if fake:
        return os.path.join(fake, kid.replace("/", "__"))
    import kagglehub
    return kagglehub.dataset_download(kid)


def _load_one(p, S, draft):
    from PIL import Image
    try:
        with Image.open(p) as im:
            if draft:
                im.draft("RGB", (S * 2, S * 2))       # fast DCT-scaled JPEG decode
            return np.asarray(im.convert("RGB").resize((S, S), Image.BILINEAR))
    except Exception as e:
        print("   unreadable, stored as black frame:", os.path.basename(p), e)
        return np.zeros((S, S, 3), np.uint8)


def load_images(paths, S=224, draft=False, workers=8):
    out = np.zeros((len(paths), S, S, 3), np.uint8)
    with ThreadPoolExecutor(workers) as ex:
        for i, a in enumerate(ex.map(lambda p: _load_one(p, S, draft), paths)):
            out[i] = a
    return out


def discover_din(root):
    """Forest Fire Dataset (alik05). Identical file lists and split to paper 1."""
    from sklearn.model_selection import train_test_split
    base = glob.glob(os.path.join(root, "**", "Forest Fire Dataset"), recursive=True)[0]
    tr_dir, te_dir = os.path.join(base, "Training"), os.path.join(base, "Testing")
    pool_p, pool_y = [], []
    for cls, lab in (("nofire", 0), ("fire", 1)):
        fs = sorted(glob.glob(os.path.join(tr_dir, cls, "*.jpg")))
        pool_p += fs; pool_y += [lab] * len(fs)
    pool_y = np.array(pool_y, np.int32)
    te_p = sorted(glob.glob(os.path.join(te_dir, "*.jpg")))
    te_y = np.array([0 if os.path.basename(p).lower().startswith("nofire") else 1
                     for p in te_p], np.int32)
    tr_p, va_p, tr_y, va_y = train_test_split(pool_p, pool_y, test_size=CFG["VAL_FRAC"],
                                              random_state=CFG["SEED"], stratify=pool_y)
    return dict(tr_p=list(tr_p), va_p=list(va_p), te_p=te_p,
                tr_y=np.array(tr_y, np.int32), va_y=np.array(va_y, np.int32), te_y=te_y)


def discover_wild(root):
    """The Wildfire Dataset: labels from 'fire'/'nofire' folders, subclass = leaf folder."""
    recs = []
    for dp, dn, fn in os.walk(root):
        dn.sort()
        parts = [p.lower() for p in dp[len(root):].split(os.sep) if p]
        if any(p == "nofire" for p in parts):
            lab = 0
        elif any(p == "fire" for p in parts):
            lab = 1
        else:
            continue
        sub = parts[-1] if parts[-1] not in ("fire", "nofire") else "unspecified"
        for f in sorted(fn):
            if os.path.splitext(f)[1].lower() in IMGEXT:
                recs.append((os.path.join(dp, f), lab, sub))
    recs.sort(key=lambda r: r[0])
    return dict(paths=[r[0] for r in recs], y=np.array([r[1] for r in recs], np.int32),
                sub=np.array([r[2] for r in recs]))


def _yolo_names(root):
    try:
        import yaml
    except Exception:
        return None
    for y in glob.glob(os.path.join(root, "**", "*.yaml"), recursive=True):
        try:
            d = yaml.safe_load(open(y))
            names = d.get("names") if isinstance(d, dict) else None
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            if names:
                return [str(n).lower() for n in names]
        except Exception:
            pass
    return None


def discover_dfire(root, prefer=("test",)):
    """D-Fire (YOLO format). fire = >=1 fire box; no-fire = no boxes; smoke-only excluded."""
    lab_dirs = [d for d in glob.glob(os.path.join(root, "**", "labels"), recursive=True)
                if os.path.isdir(os.path.join(os.path.dirname(d), "images"))]
    splits = {os.path.basename(os.path.dirname(d)).lower(): d for d in lab_dirs}
    print("D-Fire splits found:", {k: len(os.listdir(v)) for k, v in splits.items()})
    chosen = [splits[k] for k in prefer if k in splits] or list(splits.values())
    names = _yolo_names(root)
    if names and "fire" in names:
        fire_id = names.index("fire")
    else:
        fire_id = 1                                   # D-Fire convention: 0 smoke, 1 fire
    recs, stats = [], dict(fire=0, none=0, smoke_only=0)
    only = {}
    for ld in chosen:
        imd = os.path.join(os.path.dirname(ld), "images")
        imgs = {os.path.splitext(f)[0]: os.path.join(imd, f) for f in os.listdir(imd)
                if os.path.splitext(f)[1].lower() in IMGEXT}
        for stem, ip in sorted(imgs.items()):
            lf = os.path.join(ld, stem + ".txt")
            cls = []
            if os.path.exists(lf):
                cls = [int(float(l.split()[0])) for l in open(lf) if l.strip()]
            s = set(cls)
            if len(s) == 1:
                only[next(iter(s))] = only.get(next(iter(s)), 0) + 1
            if fire_id in s:
                recs.append((ip, 1, "fire+smoke" if len(s) > 1 else "fire_only")); stats["fire"] += 1
            elif not s:
                recs.append((ip, 0, "none")); stats["none"] += 1
            else:
                stats["smoke_only"] += 1
    print("D-Fire class names:", names, "-> fire id =", fire_id,
          "| images with a single class:", only, "| kept:", stats)
    if only and len(only) == 2 and only.get(fire_id, 0) > only.get(1 - fire_id, 0):
        print("WARNING: the class treated as 'fire' is the more frequent single class; in "
              "D-Fire smoke-only images outnumber fire-only images ~5:1. Check the mapping!")
    return dict(paths=[r[0] for r in recs], y=np.array([r[1] for r in recs], np.int32),
                sub=np.array([r[2] for r in recs]), stats=stats, fire_id=fire_id)


# ---------------------------------------------------------------------------
# 4. Augmentation (paper-1 recipe) and confounder-aware augmentation (CAD)
# ---------------------------------------------------------------------------
def _tf():
    import tensorflow as tf
    return tf


def std_aug_block(X, copy_idx, indices=None, batch=256):
    """Paper-1 augmentation, seeds s=[i, c*7919+13]; returns uint8 array.
    Copies 1..4 are bit-identical to the paper-1 teacher training pool."""
    tf = _tf()
    idx = np.arange(len(X)) if indices is None else np.asarray(indices)
    c = int(copy_idx)

    def aug_one(x_u8, i):
        s = tf.stack([tf.cast(i, tf.int32), tf.constant(c * 7919 + 13, tf.int32)])
        im = tf.cast(x_u8, tf.float32) / 255.0
        im = tf.image.stateless_random_flip_left_right(im, s)
        im = tf.image.stateless_random_brightness(im, 0.10, s + 1)
        im = tf.image.stateless_random_contrast(im, 0.9, 1.1, s + 2)
        im = tf.image.stateless_random_saturation(im, 0.9, 1.1, s + 3)
        im = tf.image.stateless_random_hue(im, CFG["HUE"], s + 4)
        im = tf.clip_by_value(im, 0.0, 1.0)
        im = tf.image.stateless_random_jpeg_quality(im, 75, 100, s + 5)
        return tf.cast(tf.round(im * 255.0), tf.uint8)

    ds = tf.data.Dataset.from_tensor_slices((X[idx], idx.astype(np.int32)))
    ds = ds.map(aug_one, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True).batch(batch)
    with tf.device("/CPU:0"):
        return np.concatenate([b.numpy() for b in ds]) if len(idx) else X[:0].copy()


# ---- confounder transforms (NumPy/SciPy only, deterministic per image) ----
def _warm_cast(im, r):
    """Sunset / golden-hour illumination: channel gains plus a warm sky gradient."""
    g = np.array([r.uniform(1.05, 1.35), r.uniform(0.92, 1.08), r.uniform(0.60, 0.88)])
    out = im * g
    H = im.shape[0]
    ramp = np.linspace(1.0, 0.0, H)[:, None, None] ** r.uniform(0.8, 2.0)
    a = r.uniform(0.15, 0.45) * ramp
    tint = np.array([1.0, r.uniform(0.45, 0.65), r.uniform(0.10, 0.30)])
    return out * (1 - a) + tint * a


def _foliage(im, r):
    """Autumn foliage: rotate green hues towards yellow/red, boost saturation.
    Pure NumPy (matplotlib's HSV conversion); no OpenCV dependency."""
    from matplotlib.colors import rgb_to_hsv, hsv_to_rgb
    hsv = rgb_to_hsv(np.clip(im, 0.0, 1.0).astype(np.float64))
    h = hsv[..., 0] * 360.0                                         # hue in degrees
    green = (h > 55) & (h < 170)
    h[green] = (h[green] - r.uniform(40, 95)) % 360.0
    hsv[..., 0] = h / 360.0
    hsv[..., 1] = np.clip(hsv[..., 1] * np.where(green, r.uniform(1.0, 1.3), 1.0), 0, 1)
    return hsv_to_rgb(hsv)


def _haze(im, r):
    """Fog / haze via the atmospheric scattering model I = J t + A (1 - t)."""
    H = im.shape[0]
    t0 = r.uniform(0.45, 0.85)
    grad = np.linspace(r.uniform(0.0, 0.25), 0.0, H)[:, None, None]  # denser near the top
    t = np.clip(t0 - grad, 0.25, 1.0)
    A = r.uniform(0.75, 0.95) * np.array([1.0, 1.0, r.uniform(0.95, 1.05)])
    return im * t + A * (1 - t)


def _glare(im, r):
    """Sun glare / specular reflection: additive warm Gaussian blob."""
    H, W = im.shape[:2]
    cy, cx = r.uniform(0.05, 0.95) * H, r.uniform(0.05, 0.95) * W
    s = r.uniform(0.08, 0.25) * H
    yy, xx = np.mgrid[0:H, 0:W]
    g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s * s))[..., None]
    col = np.array([1.0, r.uniform(0.80, 0.95), r.uniform(0.50, 0.75)])
    return im + r.uniform(0.3, 0.8) * g * col


def _line_kernel(L, angle_deg):
    """Normalised L x L linear motion-blur kernel at the given angle (bilinear splatting)."""
    k = np.zeros((L, L), np.float64)
    c = (L - 1) / 2.0
    t = np.linspace(-c, c, 8 * L)
    a = np.deg2rad(angle_deg)
    xs, ys = c + t * np.cos(a), c - t * np.sin(a)
    x0, y0 = np.floor(xs).astype(int), np.floor(ys).astype(int)
    fx, fy = xs - x0, ys - y0
    for dx, dy, w in ((0, 0, (1 - fx) * (1 - fy)), (1, 0, fx * (1 - fy)),
                      (0, 1, (1 - fx) * fy), (1, 1, fx * fy)):
        np.add.at(k, (np.clip(y0 + dy, 0, L - 1), np.clip(x0 + dx, 0, L - 1)), w)
    return k / k.sum()


def _motion(im, r):
    """Camera/platform motion blur: linear kernel of random length and angle.
    SciPy convolution with reflected borders; no OpenCV dependency."""
    from scipy.ndimage import convolve
    L = int(r.choice([5, 7, 9, 11, 13, 15]))
    k = _line_kernel(L, r.uniform(0, 180))
    return np.stack([convolve(im[..., ch].astype(np.float64), k, mode="reflect")
                     for ch in range(im.shape[-1])], axis=-1)


def _lowlight(im, r):
    """Dusk / low light: gamma darkening, slight blue shift and sensor noise."""
    out = im ** r.uniform(1.6, 2.6)
    out = out * np.array([r.uniform(0.85, 1.0), r.uniform(0.9, 1.0), 1.0])
    return out + r.normal(0, r.uniform(0.01, 0.04), im.shape)


CONF_OPS = [("warm_cast", _warm_cast), ("autumn_foliage", _foliage), ("haze", _haze),
            ("glare", _glare), ("motion_blur", _motion), ("low_light", _lowlight)]


def conf_one(img_u8, i, k, force=None):
    """Label-preserving confounder transform of one image (i = index, k = copy)."""
    r = np.random.default_rng([int(i), int(k), 2026])
    im = img_u8.astype(np.float32) / 255.0
    if force is not None:
        ops = [dict(CONF_OPS)[force]]
    else:
        n = 1 if r.uniform() < 0.6 else 2
        ops = [CONF_OPS[j][1] for j in r.choice(len(CONF_OPS), n, replace=False)]
    for f in ops:
        im = np.clip(f(im, r), 0.0, 1.0)
    return np.round(im * 255.0).astype(np.uint8)


def conf_aug_block(X, copy_idx, indices=None, workers=8):
    """Confounder copy k: paper-1 augmentation (seed family 100+k) then 1-2 confounders."""
    idx = np.arange(len(X)) if indices is None else np.asarray(indices)
    base = std_aug_block(X, 100 + int(copy_idx), idx)
    out = np.empty_like(base)
    with ThreadPoolExecutor(workers) as ex:
        for j, a in enumerate(ex.map(lambda t: conf_one(t[1], t[0], copy_idx),
                                     zip(idx, base))):
            out[j] = a
    return out


def pool_blocks():
    """Ordered list of augmentation blocks forming the full student pool."""
    return ([("orig", 0)] + [("std", c) for c in range(1, CFG["STD_COPIES"] + 1)]
            + [("conf", k) for k in range(1, CFG["CONF_COPIES"] + 1)])


def regime_blocks(kind):
    """Blocks used by a student regime. Both regimes have the same number of images."""
    if kind == "std":
        return [("orig", 0)] + [("std", c) for c in range(1, CFG["STD_COPIES"] + 1)]
    half = CFG["STD_COPIES"] - CFG["CONF_COPIES"]
    return ([("orig", 0)] + [("std", c) for c in range(1, half + 1)]
            + [("conf", k) for k in range(1, CFG["CONF_COPIES"] + 1)])


def teacher_blocks():
    return [("orig", 0)] + [("std", c) for c in range(1, CFG["TEACHER_COPIES"] + 1)]


def make_block(X, kind, c, indices=None):
    if kind == "orig":
        return X if indices is None else X[np.asarray(indices)]
    if kind == "std":
        return std_aug_block(X, c, indices)
    return conf_aug_block(X, c, indices)


def array_md5(a):
    return hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()


def fingerprint(a, n=20000):
    """MD5 plus a fixed random sample of pixel values (to quantify tiny cross-machine drift)."""
    flat = np.ascontiguousarray(a).reshape(-1)
    idx = np.random.default_rng(0).integers(0, flat.size, n)
    return dict(md5=array_md5(flat), sample=flat[idx].astype(int).tolist())


def compare_fingerprint(a, fp):
    flat = np.ascontiguousarray(a).reshape(-1)
    if array_md5(flat) == fp["md5"]:
        return dict(identical=True, frac_diff=0.0, max_diff=0)
    idx = np.random.default_rng(0).integers(0, flat.size, len(fp["sample"]))
    d = np.abs(flat[idx].astype(int) - np.asarray(fp["sample"]))
    return dict(identical=False, frac_diff=float((d > 0).mean()), max_diff=int(d.max()))


# ---------------------------------------------------------------------------
# 5. Perceptual-hash leakage audit
# ---------------------------------------------------------------------------
def phash_array(X):
    import imagehash
    from PIL import Image
    return np.array([int(str(imagehash.phash(Image.fromarray(x))), 16) for x in X],
                    dtype=np.uint64)


def min_hamming(a, b, chunk=512):
    """For each hash in a, the minimum Hamming distance to any hash in b."""
    out = np.empty(len(a), np.int32)
    b = b.astype(np.uint64)
    for s in range(0, len(a), chunk):
        x = np.bitwise_xor(a[s:s + chunk, None].astype(np.uint64), b[None, :])
        d = np.unpackbits(x.view(np.uint8).reshape(x.shape + (8,)), axis=-1).sum(-1)
        out[s:s + chunk] = d.min(1)
    return out


# ---------------------------------------------------------------------------
# 6. Teacher (MWAMNet) — identical architecture and protocol to paper 1
# ---------------------------------------------------------------------------
def keras_mod():
    os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
    import tf_keras as keras
    return keras


def teacher_backbones(weights="imagenet"):
    keras = keras_mod()
    if CFG["TEST_MODE"]:
        weights = None
        set_seed(0)                      # deterministic random backbones in test mode
    S = CFG["IMG"]
    d = keras.applications.DenseNet201(include_top=False, weights=weights,
                                       input_shape=(S, S, 3), pooling="avg")
    m = keras.applications.MobileNetV3Large(include_top=False, weights=weights,
                                            input_shape=(S, S, 3), pooling="avg")
    d.trainable = False; m.trainable = False
    return d, m


def densenet_prep(x):
    """== keras.applications.densenet.preprocess_input (mode 'torch')."""
    tf = _tf()
    x = tf.cast(x, tf.float32) / 255.0
    return (x - tf.constant([0.485, 0.456, 0.406])) / tf.constant([0.229, 0.224, 0.225])


def mnv3_prep(x):
    """== keras.applications.mobilenet_v3.preprocess_input (identity; model rescales)."""
    return _tf().cast(x, "float32")


def extract_features(net, prep, X, bs=64):
    tf = _tf()
    out = np.empty((len(X), net.output_shape[-1]), np.float32)
    i = 0
    while i < len(X):
        try:
            b = X[i:i + bs]
            out[i:i + len(b)] = net(prep(b), training=False).numpy()
            i += len(b)
        except tf.errors.ResourceExhaustedError:
            if bs == 1:
                raise
            bs //= 2; gc.collect()
    return out


def build_head(dims=(1920, 960), l2v=None, drop=None, units=(1024, 512)):
    """Paper-1 head. Final Dense is split into 'logit' + sigmoid (mathematically identical)."""
    keras = keras_mod(); L = keras.layers
    l2v = CFG["L2"] if l2v is None else l2v
    drop = CFG["DROPOUT"] if drop is None else drop
    ins = [keras.Input(shape=(d,)) for d in dims]
    br = [L.Dropout(drop)(L.BatchNormalization()(i)) for i in ins]
    x = L.Concatenate()(br) if len(br) > 1 else br[0]
    for u in units:
        x = L.Dense(u, activation="relu", kernel_regularizer=keras.regularizers.l2(l2v))(x)
        x = L.BatchNormalization()(x)
        x = L.Dropout(drop)(x)
    z = L.Dense(1, name="logit")(x)
    p = L.Activation("sigmoid", name="prob")(z)
    return keras.Model(ins, p, name="mwamnet_head")


def head_logit_model(head):
    keras = keras_mod()
    return keras.Model(head.inputs, head.get_layer("logit").output)


def train_head(Xtr, ytr, Xva, yva, seed):
    keras = keras_mod()
    set_seed(seed)
    steps = math.ceil(len(ytr) / CFG["BATCH"]) * CFG["EPOCHS"]
    lr = keras.optimizers.schedules.CosineDecay(CFG["LR"], steps, alpha=1e-6)
    m = build_head([f.shape[1] for f in Xtr])
    m.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=CFG["WD"]),
              loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=CFG["EPOCHS"],
              batch_size=CFG["BATCH"], verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(
                  monitor="val_accuracy", mode="max", patience=CFG["PATIENCE"],
                  restore_best_weights=True)])
    return m, len(h.history["loss"])


def build_teacher_e2e(head_weights, d=None, m=None):
    """End-to-end MWAMNet: raw pixels [0,255] (224x224x3) -> P(fire)."""
    keras = keras_mod(); L = keras.layers
    if d is None:
        d, m = teacher_backbones()
    inp = keras.Input((CFG["IMG"], CFG["IMG"], 3), name="pixels")
    xd = L.Rescaling(1.0 / 255.0)(inp)
    xd = L.Normalization(mean=[0.485, 0.456, 0.406],
                         variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2])(xd)
    fd = d(xd, training=False)
    fm = m(inp, training=False)
    head = build_head([d.output_shape[-1], m.output_shape[-1]])
    head.load_weights(head_weights)
    return keras.Model(inp, head([fd, fm]), name="MWAMNet")


# ---------------------------------------------------------------------------
# 7. Students (TinyMWAMNet)
# ---------------------------------------------------------------------------
def student_name(arch, alpha, res):
    return {"mnv3s": "MNv3S", "mnv3s6": "MNv3S6", "mnv1": "MNv1", "mnv2": "MNv2"}[arch] + f"-{alpha:g}-{res}"


def build_student(arch, alpha, res, weights="imagenet"):
    """Flat functional student core. Input in [-1, 1]; output = fire logit."""
    keras = keras_mod(); L = keras.layers
    if CFG["TEST_MODE"]:
        weights = None
    kw = dict(input_shape=(res, res, 3), include_top=False, weights=weights, pooling="avg")
    if arch in ("mnv3s", "mnv3s6"):
        base = keras.applications.MobileNetV3Small(alpha=alpha, minimalistic=True,
                                                   include_preprocessing=False, **kw)
        if arch == "mnv3s6":
            base = relu6_clone(base)
    elif arch == "mnv1":
        base = keras.applications.MobileNet(alpha=alpha, **kw)
    elif arch == "mnv2":
        base = keras.applications.MobileNetV2(alpha=alpha, **kw)
    else:
        raise ValueError(arch)
    x = L.Dropout(CFG["ST_DROPOUT"], name="st_dropout")(base.output)
    z = L.Dense(1, name="logit")(x)
    core = keras.Model(base.input, z, name="core_" + student_name(arch, alpha, res).replace(".", "p"))
    return core


def relu6_clone(base):
    """Same network and weights, but every unbounded ReLU becomes ReLU6 (activations capped at 6)."""
    keras = keras_mod(); L = keras.layers

    def cf(layer):
        if isinstance(layer, L.ReLU) and layer.max_value is None:
            return L.ReLU(max_value=6.0, name=layer.name)
        return layer.__class__.from_config(layer.get_config())
    new = keras.models.clone_model(base, clone_function=cf)
    new.set_weights(base.get_weights())
    n = sum(isinstance(l, L.ReLU) and l.max_value is not None for l in new.layers)
    assert n > 0, "no ReLU layer was capped"
    return new


def qat_range_warmup(qmodel, ds, steps):
    """tfmot initialises every activation range at [-6, 6] and updates it by an exponential
    moving average (decay 0.999) only in training mode. A short QAT run (~850 steps) leaves ~40 %
    of that initial value in the ranges. Running `steps` forward passes in training mode (no weight
    update; BN frozen) lets the ranges converge to the data before fine-tuning (0.999^3000 = 0.05)."""
    tf = _tf()

    @tf.function
    def fwd(x):
        qmodel(x, training=True)
    n = 0
    while n < steps:
        for xb, _ in ds:
            fwd(xb); n += 1
            if n >= steps:
                break
    return n


def set_backbone_trainable(core, flag):
    keras = keras_mod()
    for l in core.layers:
        if l.name in ("logit", "st_dropout"):
            l.trainable = True
        elif isinstance(l, keras.layers.BatchNormalization):
            l.trainable = False          # BN statistics frozen during fine-tuning
        else:
            l.trainable = flag


def kd_loss(alpha, T):
    """alpha*BCE(y, s) + (1-alpha)*T^2*BCE(sigma(t/T), sigma(s/T)); y_true=[y, t_logit]."""
    tf = _tf()

    def loss(y_true, z_s):
        y = y_true[:, 0:1]; zt = y_true[:, 1:2]
        hard = tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=z_s)
        if alpha >= 1.0:
            return tf.reduce_mean(hard)
        soft = tf.nn.sigmoid_cross_entropy_with_logits(labels=tf.sigmoid(zt / T),
                                                       logits=z_s / T) * (T * T)
        return tf.reduce_mean(alpha * hard + (1.0 - alpha) * soft)
    return loss


def to_unit(x_u8):
    """uint8 pixels -> x = (pixel - 128) / 128 in [-1, 0.992].
    With this convention the INT8 input tensor of the deployed student is exactly
    q = pixel - 128 (scale 1/128, zero point 0): one subtraction on the MCU."""
    return (x_u8.astype(np.float32) - 128.0) / 128.0


def export_wrapper(core_or_qat, res):
    """Deployed student: x = (pixel-128)/128 -> core -> sigmoid (P(fire))."""
    keras = keras_mod(); L = keras.layers
    inp = keras.Input((res, res, 3), name="pixels_centered")
    p = L.Activation("sigmoid", name="prob")(core_or_qat(inp))
    return keras.Model(inp, p)


def rep_unit(X_u8, n=None, seed=0):
    """Representative data for students (x in [-1, 0.992]); the first sample spans the full
    pixel range so that the converter derives input scale 1/128 and zero point 0 exactly."""
    n = len(X_u8) if n is None else min(n, len(X_u8))
    idx = np.random.default_rng(seed).choice(len(X_u8), n, replace=False)
    ext = np.zeros((1,) + X_u8.shape[1:], np.uint8); ext[:, ::2] = 255

    def gen():
        yield [to_unit(ext)]
        for i in idx:
            yield [to_unit(X_u8[i:i + 1])]
    return gen


def resize_u8(X, res, bs=512):
    """Antialiased bilinear downsampling of uint8 images to res x res (uint8)."""
    tf = _tf()
    if X.shape[1] == res:
        return X
    out = np.empty((len(X), res, res, 3), np.uint8)
    with tf.device("/CPU:0"):
        for s in range(0, len(X), bs):
            r = tf.image.resize(tf.cast(X[s:s + bs], tf.float32), (res, res),
                                method="bilinear", antialias=True)
            out[s:s + bs] = tf.cast(tf.clip_by_value(tf.round(r), 0, 255), tf.uint8).numpy()
    return out


# ---------------------------------------------------------------------------
# 8. TFLite conversion, LiteRT / TFLM runners, Vela
# ---------------------------------------------------------------------------
def tflite_convert(model, res, mode, rep=None, in_type="uint8"):
    """mode: fp32 | fp16 | dyn | int8 (full-integer; input uint8 (teacher, raw pixels) or
    int8 (students, pixel-128); output uint8 probability, scale 1/256)."""
    tf = _tf()

    @tf.function(input_signature=[tf.TensorSpec([1, res, res, 3], tf.float32)])
    def serve(x):
        return model(x, training=False)

    c = tf.lite.TFLiteConverter.from_concrete_functions([serve.get_concrete_function()], model)
    if mode == "fp16":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.target_spec.supported_types = [tf.float16]
    elif mode == "dyn":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
    elif mode == "int8":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = rep
        c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        c.inference_input_type = tf.int8 if in_type == "int8" else tf.uint8
        c.inference_output_type = tf.uint8
    elif mode != "fp32":
        raise ValueError(mode)
    return c.convert()


def rep_from(X_u8, n=None, seed=0):
    """Representative dataset generator (float pixels in [0,255], batch 1)."""
    n = len(X_u8) if n is None else min(n, len(X_u8))
    idx = np.random.default_rng(seed).choice(len(X_u8), n, replace=False)

    def gen():
        for i in idx:
            yield [X_u8[i:i + 1].astype(np.float32)]
    return gen


LAST_BACKEND = None     # runtime used by the most recent LiteRT call (stored with every result)


def _litert():
    try:
        from ai_edge_litert import interpreter as lit
        return lit
    except Exception:
        return None


def _runtime_candidates(kernels="auto"):
    """Fallback chain. XNNPACK is fastest, but it cannot prepare some INT8 graphs
    (e.g. MobileNetV3 blocks in some TensorFlow builds): the built-in kernels always work.
    kernels="ref" keeps only the built-in (reference-equivalent) kernels, whose integer arithmetic
    matches TensorFlow Lite Micro, i.e. what the microcontroller executes."""
    import warnings
    warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")
    tf = _tf(); lit = _litert(); c = []
    if lit is not None:
        c.append(("litert-xnnpack", lit.Interpreter, {}))
    c.append(("tflite-xnnpack", tf.lite.Interpreter, {}))
    if lit is not None:
        c.append(("litert-builtin", lit.Interpreter,
                  {"experimental_op_resolver_type": lit.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES}))
    c.append(("tflite-builtin", tf.lite.Interpreter,
              {"experimental_op_resolver_type": tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES}))
    skip = [x for x in os.environ.get("TMN_SKIP_RUNTIMES", "").split(",") if x]
    if kernels == "ref":
        c = [x for x in c if x[0].endswith("builtin")]
    return [x for x in c if x[0] not in skip]


_RUNTIME_OK = {}        # model fingerprint -> runtime that worked (skips known failures)


def make_interpreter(model_bytes, threads=2, batch=None, verbose=True, kernels="auto",
                     preserve_all=False):
    """Allocated, smoke-tested interpreter; tries each runtime in turn. Returns (it, name)."""
    global LAST_BACKEND
    key = (len(model_bytes), hash(model_bytes), kernels, preserve_all)
    cands = _runtime_candidates(kernels)
    if key in _RUNTIME_OK:                    # the runtime that worked last time goes first
        cands = sorted(cands, key=lambda c: c[0] != _RUNTIME_OK[key])
    err = None
    for name, cls, kw in cands:
        try:
            kw2 = dict(kw, experimental_preserve_all_tensors=True) if preserve_all else kw
            it = cls(model_content=model_bytes, num_threads=threads, **kw2)
            ind = it.get_input_details()[0]
            if batch:
                it.resize_tensor_input(ind["index"], [batch] + list(ind["shape"][1:]))
            it.allocate_tensors()
            ind = it.get_input_details()[0]
            it.set_tensor(ind["index"], np.zeros(ind["shape"], ind["dtype"])); it.invoke()
            LAST_BACKEND = name; _RUNTIME_OK[key] = name
            return it, name
        except Exception as e:
            err = e
            if verbose:
                print("   runtime %s cannot run this model (%s) -> trying the next one"
                      % (name, str(e).split("\n")[0][:90]))
    raise err


def quantize_input(xb_u8, dtype, scale, zp):
    """Map uint8 camera pixels to the model's input tensor."""
    if dtype == np.uint8:
        if abs(scale - 1.0) < 1e-3 and zp == 0:
            return xb_u8
        return np.clip(np.round(xb_u8 / scale + zp), 0, 255).astype(np.uint8)
    if dtype == np.int8:                      # students: x = (p-128)/128
        if abs(scale - 1 / 128) < 1e-7 and zp == 0:
            return (xb_u8.astype(np.int16) - 128).astype(np.int8)
        return np.clip(np.round(to_unit(xb_u8) / scale + zp), -128, 127).astype(np.int8)
    return xb_u8.astype(np.float32)           # float models (teacher FP32/FP16/dynamic)


def tflite_predict(model_bytes, X_u8, bs=16, threads=2, kernels="auto"):
    """Batched LiteRT/TFLite inference; returns P(fire) as float32 (runtime in LAST_BACKEND).
    kernels="ref": built-in reference-equivalent kernels only (matches the MCU arithmetic)."""
    it, _ = make_interpreter(model_bytes, threads, batch=bs, kernels=kernels)
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    in_s, in_z = ind["quantization"]; out_s, out_z = outd["quantization"]
    out = np.empty(len(X_u8), np.float32)
    for s in range(0, len(X_u8), bs):
        xb = X_u8[s:s + bs]
        n = len(xb)
        if n < bs:
            xb = np.concatenate([xb, np.zeros((bs - n,) + xb.shape[1:], xb.dtype)])
        xin = quantize_input(xb, ind["dtype"], in_s, in_z)
        it.set_tensor(ind["index"], xin); it.invoke()
        o = it.get_tensor(outd["index"]).reshape(bs, -1)[:n, 0]
        out[s:s + n] = (o.astype(np.float32) - out_z) * out_s if outd["dtype"] != np.float32 else o
    return out


def tflite_latency(model_bytes, threads=1, warm=50, runs=200):
    """Median single-image latency (ms) with the first runtime that can run the model."""
    it, _ = make_interpreter(model_bytes, threads, batch=None)
    ind = it.get_input_details()[0]
    x = np.random.default_rng(0).integers(0, 255, ind["shape"]).astype(ind["dtype"])
    for _ in range(warm):
        it.set_tensor(ind["index"], x); it.invoke()
    ts = []
    for _ in range(runs):
        t = time.perf_counter(); it.set_tensor(ind["index"], x); it.invoke()
        ts.append(time.perf_counter() - t)
    return float(np.median(ts) * 1e3)


_TFLM_PROBE = r'''
import sys, numpy as np
from tflite_micro.python.tflite_micro import runtime as r
buf = open(sys.argv[1], "rb").read()
it = r.Interpreter.from_bytes(buf)
d = it.get_input_details(0)
it.set_input(np.zeros(d["shape"], d["dtype"]), 0); it.invoke(); it.print_allocations()
'''


def tflm_arena_bytes(tflite_path):
    """Exact tensor-arena size recorded by the TFLite Micro RecordingMicroAllocator."""
    probe = "/tmp/_tmn_tflm_probe.py"
    open(probe, "w").write(_TFLM_PROBE)
    r = subprocess.run([sys.executable, probe, tflite_path], capture_output=True, text=True)
    m = re.search(r"Arena allocation total (\d+)", r.stdout + r.stderr)
    return int(m.group(1)) if m else None


def tflm_predict_u8(tflite_path, X_u8):
    """Run the TFLite Micro reference runtime (the MCU arithmetic) on uint8 images."""
    from tflite_micro.python.tflite_micro import runtime as r
    it = r.Interpreter.from_bytes(open(tflite_path, "rb").read())
    d = it.get_input_details(0); q = d["quantization_parameters"]
    s, z = float(q["scales"][0]), int(q["zero_points"][0])
    out = np.empty(len(X_u8), np.int32)
    for i in range(len(X_u8)):
        it.set_input(quantize_input(X_u8[i:i + 1], d["dtype"], s, z), 0); it.invoke()
        out[i] = int(np.asarray(it.get_output(0)).ravel()[0])
    return out


def tflite_raw_u8(model_bytes, X_u8, kernels="auto"):
    it, _ = make_interpreter(model_bytes, 2, batch=None, kernels=kernels)
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    s, z = ind["quantization"]
    out = np.empty(len(X_u8), np.int32)
    for i in range(len(X_u8)):
        it.set_tensor(ind["index"], quantize_input(X_u8[i:i + 1], ind["dtype"], s, z)); it.invoke()
        out[i] = int(it.get_tensor(outd["index"]).ravel()[0])
    return out


def _vela_cmd():
    exe = shutil.which("vela") or os.path.join(os.path.dirname(sys.executable), "vela")
    return [exe] if os.path.exists(exe) else [sys.executable, "-m", "ethosu.vela"]


def vela_estimate(tflite_path, outdir, accel="ethos-u55-128"):
    outdir = os.path.join(outdir, accel)          # one folder per accelerator configuration
    shutil.rmtree(outdir, ignore_errors=True)
    os.makedirs(outdir, exist_ok=True)
    try:
        r = subprocess.run(_vela_cmd() + [tflite_path, "--accelerator-config", accel,
                                          "--output-dir", outdir], capture_output=True, text=True)
    except OSError as e:
        print("Vela could not be started (%s); the NPU estimate is skipped." % e)
        return None
    stem = os.path.splitext(os.path.basename(tflite_path))[0]
    cs = glob.glob(os.path.join(outdir, stem + "_summary_*.csv"))
    if not cs:
        print("Vela failed:", r.stdout[-800:], r.stderr[-800:])
        return None
    import csv
    row = list(csv.DictReader(open(cs[0])))[0]
    return dict(vela_accel=accel, vela_ms=float(row["inference_time"]) * 1e3,
                vela_sram_KiB=float(row["sram_memory_used"]),
                vela_flash_KiB=float(row.get("off_chip_flash_memory_used", 0) or 0),
                vela_macs=int(float(row["nn_macs"])), vela_cycles=int(float(row["cycles_total"])))


def count_macs(model, res):
    """Multiply-accumulate count of the float graph (TF profiler FLOPs / 2)."""
    tf = _tf()
    from tensorflow.python.framework.convert_to_constants import \
        convert_variables_to_constants_v2_as_graph
    f = tf.function(lambda x: model(x, training=False)).get_concrete_function(
        tf.TensorSpec([1, res, res, 3], tf.float32))
    frozen, _ = convert_variables_to_constants_v2_as_graph(f)
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    opts["output"] = "none"
    prof = tf.compat.v1.profiler.profile(graph=frozen.graph, run_meta=tf.compat.v1.RunMetadata(),
                                         cmd="op", options=opts)
    return int(prof.total_float_ops // 2)


# ---------------------------------------------------------------------------
# 9. Weight-only fake quantisation (bit-width sweep)
# ---------------------------------------------------------------------------
def _iter_layers(model):
    keras = keras_mod()
    for l in model.layers:
        if isinstance(l, keras.Model):
            yield from _iter_layers(l)
        else:
            yield l


def fake_quant_weights(model, bits):
    """Per-output-channel symmetric b-bit quantisation of every Conv/Depthwise/Dense
    kernel, in place. Returns a backup list for restore_weights()."""
    keras = keras_mod()
    qmax = 2 ** (bits - 1) - 1
    backup = []
    for l in _iter_layers(model):
        if isinstance(l, keras.layers.DepthwiseConv2D):
            axes = (0, 1)                      # kernel (kh, kw, channels, multiplier)
        elif isinstance(l, (keras.layers.Conv2D, keras.layers.Dense)):
            axes = tuple(range(len(l.kernel.shape) - 1))   # per output channel
        else:
            continue
        var = l.depthwise_kernel if isinstance(l, keras.layers.DepthwiseConv2D) else l.kernel
        W = var.numpy()
        backup.append((var, W.copy()))
        s = np.maximum(np.abs(W).max(axis=axes, keepdims=True), 1e-12) / qmax
        var.assign(np.clip(np.round(W / s), -qmax, qmax) * s)
    return backup


def restore_weights(backup):
    for var, W in backup:
        var.assign(W)


# ---------------------------------------------------------------------------
# 10. Metrics and statistics
# ---------------------------------------------------------------------------
def ece_score(y, p, bins=15):
    """Expected calibration error of P(fire) (binary, equal-width bins)."""
    y = np.asarray(y); p = np.asarray(p, float)
    e = np.linspace(0, 1, bins + 1); tot = 0.0
    for a, b in zip(e[:-1], e[1:]):
        m = (p > a) & (p <= b) if a > 0 else (p >= a) & (p <= b)
        if m.any():
            tot += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(tot)


def metrics(y, p, thr=0.5):
    from sklearn.metrics import roc_auc_score, average_precision_score
    y = np.asarray(y).astype(int); p = np.asarray(p, float); yp = (p > thr).astype(int)
    tp = int(((yp == 1) & (y == 1)).sum()); tn = int(((yp == 0) & (y == 0)).sum())
    fp = int(((yp == 1) & (y == 0)).sum()); fn = int(((yp == 0) & (y == 1)).sum())
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    two = len(np.unique(y)) == 2
    return dict(n=len(y), acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, recall=rec,
                specificity=spec, precision=prec,
                f1=2 * prec * rec / max(prec + rec, 1e-12),
                auc=roc_auc_score(y, p) if two else np.nan,
                ap=average_precision_score(y, p) if two else np.nan,
                ece=ece_score(y, p), brier=float(np.mean((p - y) ** 2)),
                tp=tp, tn=tn, fp=fp, fn=fn)


def wilson(k, n, z=1.959964):
    if n == 0:
        return (np.nan, np.nan)
    ph = k / n; d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


def mcnemar_p(y, pa, pb, thr=0.5):
    """Two-sided McNemar test on paired correctness (exact if discordant < 25)."""
    from statsmodels.stats.contingency_tables import mcnemar
    a = ((np.asarray(pa) > thr).astype(int) == y); b = ((np.asarray(pb) > thr).astype(int) == y)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    if n01 + n10 == 0:
        return 1.0, n01, n10
    r = mcnemar([[int((a & b).sum()), n01], [n10, int((~a & ~b).sum())]],
                exact=(n01 + n10) < 25, correction=True)
    return float(r.pvalue), n01, n10


def holm(pvals):
    from statsmodels.stats.multitest import multipletests
    return multipletests(pvals, method="holm")[1]


def flips(y, p_ref, p_new, thr=0.5):
    a = (np.asarray(p_ref) > thr).astype(int); b = (np.asarray(p_new) > thr).astype(int)
    f = a != b
    return dict(flip_rate=float(f.mean()), n_flip=int(f.sum()),
                new_missed_fires=int((f & (y == 1) & (b == 0)).sum()),
                new_false_alarms=int((f & (y == 0) & (b == 1)).sum()),
                fixed=int((f & (b == y)).sum()), broken=int((f & (a == y)).sum()))


def paired_bootstrap(y, pa, pb, fn="bacc", B=2000, seed=0, thr=0.5):
    """95% CI of metric(pb) - metric(pa) by paired bootstrap over images."""
    rng = np.random.default_rng(seed); y = np.asarray(y); n = len(y)
    f = (lambda yy, pp: metrics(yy, pp, thr)[fn])
    d0 = f(y, pb) - f(y, pa); ds = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2:
            continue
        ds.append(f(y[i], pb[i]) - f(y[i], pa[i]))
    lo, hi = np.percentile(ds, [2.5, 97.5])
    return float(d0), float(lo), float(hi)


# ---------------------------------------------------------------------------
# 11. Result bookkeeping (resumable)
# ---------------------------------------------------------------------------
def labels(P):
    return dict(np.load(os.path.join(P["data"], "labels.npz"), allow_pickle=True))


def teacher_soft(P):
    """Teacher-ensemble soft labels (logits) for every pool block: {'orig_0': array, ...}."""
    d = np.load(os.path.join(P["data"], "teacher_soft.npz"))
    return {k: d[k] for k in d.files}


def pred_path(P, model_id):
    return os.path.join(P["preds"], model_id + ".npz")


def have_preds(P, model_id):
    return os.path.exists(pred_path(P, model_id))


def save_preds(P, model_id, preds, meta, idx=None):
    """Per-image P(fire). If idx is given, preds[ds] belongs to images idx[ds] of set ds."""
    extra = {"idx_" + k: np.asarray(v, np.int32) for k, v in (idx or {}).items()}
    np.savez_compressed(pred_path(P, model_id), meta=json.dumps(meta), **extra,
                        **{k: np.asarray(v, np.float32) for k, v in preds.items()})


def load_preds(P, model_id):
    """Returns (preds, meta, idx); idx[ds] is None when the full set was evaluated."""
    d = np.load(pred_path(P, model_id), allow_pickle=True)
    preds = {k: d[k] for k in d.files if k != "meta" and not k.startswith("idx_")}
    idx = {k: (d["idx_" + k] if "idx_" + k in d.files else None) for k in preds}
    return preds, json.loads(str(d["meta"])), idx


def metric_rows(P, model_id, preds, meta, idx=None):
    lab = labels(P); rows = []
    for ds in EVALSETS:
        if ds in preds:
            y = lab["y_" + ds] if not idx or idx.get(ds) is None else lab["y_" + ds][idx[ds]]
            r = dict(meta); r.update(model_id=model_id, dataset=ds)
            r.update(metrics(y, preds[ds])); rows.append(r)
    return rows


def ladder_subsets(P, n=1000, seed=42):
    """Fixed evaluation subsets for the teacher's TFLite ladder: the whole in-domain test set
    and a label-stratified random sample of n images from each shifted set.
    A per-class random order is drawn once and saved (data/ladder_order.json); the subset of
    size n is the first round(n * class share) images of each class, so subsets for different
    n are nested (a 600-image subset lies inside the 1,000-image one)."""
    f = os.path.join(P["data"], "ladder_order.json")
    lab = labels(P)
    if os.path.exists(f):
        order = {k: np.array(v, np.int64) for k, v in load_json(f).items()}
    else:
        rng = np.random.default_rng(seed); order = {}
        for ds in EVALSETS:
            y = lab["y_" + ds]
            for c in (0, 1):
                order[f"{ds}_{c}"] = rng.permutation(np.where(y == c)[0])
        save_json({k: v.tolist() for k, v in order.items()}, f)
    out = {}
    for ds in EVALSETS:
        y = lab["y_" + ds]
        if ds == "din" or len(y) <= n:
            out[ds] = np.arange(len(y)); continue
        pick = [order[f"{ds}_{c}"][:int(round(n * np.sum(y == c) / len(y)))] for c in (0, 1)]
        out[ds] = np.sort(np.concatenate(pick))
    return out


def common_index(idx_list, n_total):
    """Images shared by several prediction sets (None = the full set)."""
    keep = np.ones(n_total, bool)
    for ix in idx_list:
        if ix is not None:
            m = np.zeros(n_total, bool); m[np.asarray(ix)] = True; keep &= m
    return np.where(keep)[0]


def preds_on(pr_ds, ix_ds, target):
    """Predictions of one model restricted to the image indices in `target`."""
    if ix_ds is None:
        return np.asarray(pr_ds)[target]
    pos = {int(i): k for k, i in enumerate(np.asarray(ix_ds))}
    return np.asarray(pr_ds)[[pos[int(i)] for i in target]]


def append_rows(path, rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    if os.path.exists(path):
        old = pd.read_csv(path)
        old = old[~old["model_id"].isin(df["model_id"].unique())]
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(path, index=False)
    return df


def load_metrics(P):
    """All metric rows from every notebook (each notebook writes its own CSV)."""
    import pandas as pd
    fs = sorted(glob.glob(os.path.join(P["results"], "metrics_nb*.csv")))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


def load_eval_arrays(P, res=None):
    """Evaluation images at 224 (or resized to res), keyed by EVALSETS."""
    d = np.load(os.path.join(P["data"], "eval_224.npz"))
    X = {k: d["X_" + k] for k in EVALSETS}
    if res and res != CFG["IMG"]:
        X = {k: resize_u8(v, res) for k, v in X.items()}
    return X


def env_report():
    rep = dict(python=sys.version.split()[0])
    for pkg in ["tensorflow", "tf_keras", "tensorflow_model_optimization", "numpy",
                "tflite_micro", "ethos-u-vela", "kagglehub", "imagehash"]:
        try:
            import importlib.metadata as md
            rep[pkg] = md.version(pkg.replace("_", "-") if pkg != "tflite_micro" else "tflite-micro")
        except Exception:
            try:
                rep[pkg] = md.version(pkg)
            except Exception:
                rep[pkg] = None
    try:
        tf = _tf()
        rep["gpu"] = [d.name for d in tf.config.list_physical_devices("GPU")]
    except Exception:
        pass
    return rep

Writing /content/tmn_common.py


In [ ]:
#@title 3 · Mount Google Drive and load the shared code
import os, sys, importlib
os.environ["TF_USE_LEGACY_KERAS"] = "1"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not mounted (not on Colab?):", e)
sys.path.insert(0, "/content")
import tmn_common as C; importlib.reload(C)
P = C.setup_paths()
import numpy as np, pandas as pd, tensorflow as tf
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
import tf_keras as keras
print("Notebook version:", C.VERSION)
ENV = C.env_report(); print(ENV)
C.save_json(ENV, os.path.join(P["logs"], "env_nb5.json"))
print("All outputs are saved under:", P["root"])
if False and not tf.config.list_physical_devices("GPU") and not C.CFG["TEST_MODE"]:
    raise SystemExit("No GPU. In Colab: Runtime > Change runtime type > T4 GPU, then re-run.")

Mounted at /content/drive
Notebook version: 30 Sep 2026
{'python': '3.13.15', 'tensorflow': '2.20.0', 'tf_keras': '2.20.1', 'tensorflow_model_optimization': None, 'numpy': '2.1.3', 'tflite_micro': '0.dev20251201172741', 'ethos-u-vela': None, 'kagglehub': '1.0.2', 'imagehash': None, 'gpu': []}
All outputs are saved under: /content/drive/MyDrive/TinyMWAMNet


In [ ]:
#@title 4 · Labels, evaluation images, metrics and model bookkeeping
import time, gc
lab = C.labels(P); X224 = C.load_eval_arrays(P)
A = os.path.join(P["results"], "analysis"); os.makedirs(os.path.join(A, "overflow"), exist_ok=True)
M = C.load_metrics(P)
_ev = {}
def ev(res):
    if res not in _ev:
        _ev.clear(); gc.collect()
        _ev[res] = {k: C.resize_u8(v, res) for k, v in X224.items()}
    return _ev[res]
STUDENTS = [C.MAIN_STUDENT, C.RELU6_STUDENT] + C.FAMILY
NAME = {C.student_name(*a): a for a in STUDENTS}
def regimes_of(name):
    return list(C.REGIMES) if name == C.student_name(*C.MAIN_STUDENT) else ["CE-std", "CAD"]
def seeds_with(name, regime, kind="fp32"):
    return sorted(s for s in C.CFG["SEEDS"] if C.have_preds(P, pid(name, regime, s, kind)))
def pid(name, regime, seed, kind):
    base = f"S_{name}_{regime}_s{seed}"
    return {"fp32": base + "_fp32", "ptq": base + "_ptq_ref", "qat": base + "_qatw_ref",
            "ptq_xnn": base + "_ptq", "qat_old": base + "_qat"}[kind]
def tfl(name, regime, seed, kind):
    return os.path.join(P["tflite"], f"S_{name}_{regime}_s{seed}" + {"ptq": "_ptq", "qat": "_qatw"}[kind] + ".tflite")
def median_seed(name, regime):
    f = M[(M.arch == name) & (M.regime == regime) & (M.quant == "fp32") & (M.dataset == "din")]
    f = f.drop_duplicates("seed").sort_values(["acc", "seed"])
    return int(f.iloc[len(f) // 2]["seed"]) if len(f) else None
def preds(mid):
    pr, meta, idx = C.load_preds(P, mid)
    return pr, meta, idx
Y = {d: lab["y_" + d] for d in C.EVALSETS}
avail = [(n, r, s) for n in NAME for r in regimes_of(n) for s in seeds_with(n, r)]
print("student runs with FP32 predictions:", len(avail))
print("median seeds:", {f"{n}/{r}": median_seed(n, r) for n in NAME for r in regimes_of(n) if seeds_with(n, r)})

student runs with FP32 predictions: 54
median seeds: {'MNv3S-1-128/CE-std': 44, 'MNv3S-1-128/CE-conf': 46, 'MNv3S-1-128/KD-std': 42, 'MNv3S-1-128/CAD': 44, 'MNv3S6-1-128/CE-std': 44, 'MNv3S6-1-128/CAD': 46, 'MNv1-0.25-96/CE-std': 42, 'MNv1-0.25-96/CAD': 44, 'MNv3S-1-96/CE-std': 44, 'MNv3S-1-96/CAD': 44, 'MNv3S-1-160/CE-std': 44, 'MNv3S-1-160/CAD': 43, 'MNv2-0.35-128/CE-std': 42, 'MNv2-0.35-128/CAD': 42}


In [ ]:
#@title 5 · Runtime agreement: reference kernels vs TensorFlow Lite Micro vs XNNPACK
f_out = os.path.join(A, "runtime_agreement.csv")
rows = []
# (a) every evaluation image: XNNPACK scores (Notebook 03) vs reference-kernel scores (Notebook 04), PTQ students
for n, r, s in avail:
    a_id, b_id = pid(n, r, s, "ptq_xnn"), pid(n, r, s, "ptq")
    if C.have_preds(P, a_id) and C.have_preds(P, b_id):
        a, b = preds(a_id)[0], preds(b_id)[0]
        for d in C.EVALSETS:
            rows.append(dict(model_id=a_id, arch=n, regime=r, seed=s, dataset=d, n=len(a[d]),
                             comparison="xnnpack_vs_ref", decision_agreement=float(((a[d] > .5) == (b[d] > .5)).mean()),
                             max_abs_prob_diff=float(np.abs(a[d] - b[d]).max())))
# (b) raw INT8 outputs on 48 images (24 in-domain, 12 + 12 shifted), representative models.
#     TFLM's Python runtime uses slow reference loops (~1 s per image on Colab), hence the small sample.
rng = np.random.default_rng(0); NCHK = 48 if not C.CFG["TEST_MODE"] else 12
main_like = (C.student_name(*C.MAIN_STUDENT), C.student_name(*C.RELU6_STUDENT))
for n, (arch, alpha, res) in NAME.items():
    X = ev(res)
    chk = np.concatenate([X[d][rng.choice(len(X[d]), min(k, len(X[d])), replace=False)]
                          for d, k in (("din", NCHK // 2), ("wild", NCHK // 4), ("dfire", NCHK // 4))])
    for r in (["CE-std", "CAD"] if n in main_like else ["CAD"]):
        s = median_seed(n, r)
        for kind in (("ptq", "qat") if n in main_like else ("ptq",)):
            f = tfl(n, r, s, kind) if s is not None else ""
            if not os.path.exists(f):
                continue
            b = open(f, "rb").read()
            ref = C.tflite_raw_u8(b, chk, kernels="ref")
            others = {"tflm": (C.tflm_predict_u8(f, chk), "tflite-micro")}
            xnn = C.tflite_raw_u8(b, chk, kernels="auto"); others["xnnpack"] = (xnn, C.LAST_BACKEND)
            for o, (arr, rt) in others.items():
                rows.append(dict(model_id=os.path.basename(f)[:-7], arch=n, regime=r, seed=s, dataset="mixed256",
                                 n=len(chk), comparison=f"ref_vs_{o}", other_runtime=rt, exact_match=float((ref == arr).mean()),
                                 max_abs_lsb=int(np.abs(ref.astype(int) - arr.astype(int)).max()),
                                 decision_agreement=float(((ref > 128) == (arr > 128)).mean())))
RA = pd.DataFrame(rows); RA.to_csv(f_out, index=False)
print(RA.groupby("comparison")[["decision_agreement", "exact_match", "max_abs_lsb"]]
        .agg(["mean", "min", "max"]).round(4).to_string())

/usr/local/lib/python3.13/dist-packages/tflite_micro/python/tflite_micro/postinstall_check.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
Number of resource variables the model uses =  0
               decision_agreement              exact_match                max_abs_lsb           
                             mean     min  max        mean    min     max        mean  min   max
comparison                                                                                      
ref_vs_tflm                0.9913  0.9583  1.0      0.7743  0.625  1.0000     27.8333  0.0  92.0
ref_vs_xnnpack          

In [ ]:
#@title 6 · Activation overflow: how often do INT8 activations hit the limits fixed on in-domain calibration images?
def act_tensors(it):
    td = {t["index"]: t for t in it.get_tensor_details()}
    out = []
    for o in it._get_ops_details():
        if o["op_name"] in ("QUANTIZE", "DEQUANTIZE", "LOGISTIC"):
            continue
        for ti in o["outputs"]:
            t = td[ti]
            if t["dtype"] != np.int8 or not t["quantization"][0]:
                continue
            sc, z = t["quantization"]; top, bottom = sc * (127 - z), sc * (-128 - z)
            capped = ("Relu6" in t["name"]) and top >= 5.94       # the top code is the ReLU6 cap, not an overflow
            out.append((ti, capped, bottom < -1e-9))
    return out

def overflow_scores(b, X, bs=32):
    # share of each image's INT8 activations that sit at a clipping limit (batched for speed)
    it, _ = C.make_interpreter(b, threads=2, batch=bs, kernels="ref", preserve_all=True)
    ind = it.get_input_details()[0]; sc, z = ind["quantization"]; T = act_tensors(it)
    frac = np.zeros(len(X), np.float32)
    for s0 in range(0, len(X), bs):
        xb = X[s0:s0 + bs]; n = len(xb)
        if n < bs:
            xb = np.concatenate([xb, np.zeros((bs - n,) + xb.shape[1:], xb.dtype)])
        it.set_tensor(ind["index"], C.quantize_input(xb, ind["dtype"], sc, z)); it.invoke()
        over = np.zeros(bs); tot = 0
        for ti, capped, signed in T:
            a = it.get_tensor(ti).reshape(bs, -1)
            if not capped:
                over += np.count_nonzero(a == 127, axis=1)
            if signed:
                over += np.count_nonzero(a == -128, axis=1)
            tot += a.shape[1]
        frac[s0:s0 + n] = (over / max(tot, 1))[:n]
    return frac

def overflow_model(mid, b, X, bs=32):
    f = os.path.join(A, "overflow", mid + ".npz")
    if os.path.exists(f):
        d = np.load(f); return {k: d[k] for k in d.files}
    t0 = time.time(); out = {d: overflow_scores(b, X[d], bs) for d in X}
    np.savez_compressed(f, **out); print("   %-40s %.1f min" % (mid, (time.time() - t0) / 60))
    return out

OV = []
# all seeds for the PTQ models of TinyMWAMNet-S (ReLU) and its ReLU6 version; the median seed otherwise
main_like = (C.student_name(*C.MAIN_STUDENT), C.student_name(*C.RELU6_STUDENT))
jobs = []
for n in NAME:
    for r in ("CE-std", "CAD"):
        ms_ = median_seed(n, r)
        for s in seeds_with(n, r):
            for k in (("ptq", "qat") if n in main_like else ("ptq",)):
                if (n in main_like and k == "ptq") or s == ms_:
                    jobs.append((n, r, s, k))
print(len(jobs), "INT8 student models to analyse")
for n, r, s, k in jobs:
    f = tfl(n, r, s, k)
    if not os.path.exists(f):
        continue
    X = ev(NAME[n][2]); fr = overflow_model(os.path.basename(f)[:-7], open(f, "rb").read(), X)
    p32 = preds(pid(n, r, s, "fp32"))[0]; pq = preds(pid(n, r, s, k))[0]
    for d in C.EVALSETS:
        flip = (p32[d] > .5) != (pq[d] > .5)
        for c in (0, 1):
            m = Y[d] == c
            OV.append(dict(arch=n, regime=r, seed=s, quant=k, dataset=d, label=c, n=int(m.sum()),
                           overflow_mean=float(fr[d][m].mean()), any_overflow=float((fr[d][m] > 0).mean()),
                           flip_rate=float(flip[m].mean()),
                           overflow_flipped=float(fr[d][m & flip].mean()) if (m & flip).any() else np.nan,
                           overflow_kept=float(fr[d][m & ~flip].mean()) if (m & ~flip).any() else np.nan))

RUN_TEACHER_OVERFLOW = True        # INT8 teacher on 100 fire + 100 no-fire images per dataset (~10 min)
if RUN_TEACHER_OVERFLOW:
    med = C.load_json(os.path.join(P["data"], "teacher_median_seed.json"))["median_seed"]
    f = os.path.join(P["tflite"], f"T_int8_s{med}.tflite")
    if os.path.exists(f):
        p32 = preds(f"T_fp32_s{med}")[0]; pq, _, iq = preds(f"T_int8_s{med}")
        rng = np.random.default_rng(1); pick = {}
        for d in C.EVALSETS:                   # 100 fire + 100 no-fire images from the ladder subset
            cand = iq[d] if iq.get(d) is not None else np.arange(len(Y[d])); yy = Y[d][cand]
            pick[d] = np.sort(np.concatenate([rng.choice(cand[yy == c], min(100, int((yy == c).sum())), replace=False)
                                              for c in (0, 1)]))
        fr = overflow_model(f"T_int8_s{med}_sub600", open(f, "rb").read(), {d: X224[d][pick[d]] for d in C.EVALSETS}, bs=4)
        for d in C.EVALSETS:
            a = p32[d][pick[d]] > .5; b = C.preds_on(pq[d], iq.get(d), pick[d]) > .5; flip = a != b
            for c in (0, 1):
                m = Y[d][pick[d]] == c
                OV.append(dict(arch="MWAMNet", regime="-", seed=med, quant="int8", dataset=d, label=c, n=int(m.sum()),
                               overflow_mean=float(fr[d][m].mean()), any_overflow=float((fr[d][m] > 0).mean()),
                               flip_rate=float(flip[m].mean()),
                               overflow_flipped=float(fr[d][m & flip].mean()) if (m & flip).any() else np.nan,
                               overflow_kept=float(fr[d][m & ~flip].mean()) if (m & ~flip).any() else np.nan))
OV = pd.DataFrame(OV); OV.to_csv(os.path.join(A, "overflow_summary.csv"), index=False)
if len(OV):
    print((OV.groupby(["arch", "quant", "dataset"])[["overflow_mean", "flip_rate"]].mean() * 100).round(3).to_string())

32 INT8 student models to analyse
   S_MNv3S-1-128_CE-std_s42_ptq             0.5 min
   S_MNv3S-1-128_CE-std_s43_ptq             0.5 min
   S_MNv3S-1-128_CE-std_s44_ptq             0.5 min
   S_MNv3S-1-128_CE-std_s44_qatw            0.6 min
   S_MNv3S-1-128_CE-std_s45_ptq             0.5 min
   S_MNv3S-1-128_CE-std_s46_ptq             0.5 min
   S_MNv3S-1-128_CAD_s42_ptq                0.5 min
   S_MNv3S-1-128_CAD_s43_ptq                0.5 min
   S_MNv3S-1-128_CAD_s44_ptq                0.5 min
   S_MNv3S-1-128_CAD_s44_qatw               0.5 min
   S_MNv3S-1-128_CAD_s45_ptq                0.5 min
   S_MNv3S-1-128_CAD_s46_ptq                0.5 min
   S_MNv3S6-1-128_CE-std_s42_ptq            0.5 min
   S_MNv3S6-1-128_CE-std_s43_ptq            0.5 min
   S_MNv3S6-1-128_CE-std_s44_ptq            0.5 min
   S_MNv3S6-1-128_CE-std_s44_qatw           0.5 min
   S_MNv3S6-1-128_CE-std_s45_ptq            0.5 min
   S_MNv3S6-1-128_CE-std_s46_ptq            0.4 min
   S_MNv3S6-1-128_CAD_s42_ptq 

In [ ]:
#@title 7 · Statistics: summaries, per-seed INT8 costs, McNemar–Holm, bootstrap, flips, margins
from scipy import stats as st

def bacc_boot(y, pa, pb, B=2000, seed=0):
    # paired bootstrap of balanced accuracy difference (pb - pa), vectorised
    rng = np.random.default_rng(seed); y = np.asarray(y).astype(bool); a = np.asarray(pa) > .5; b = np.asarray(pb) > .5
    n = len(y); out = []
    for _ in range(B // 500):
        i = rng.integers(0, n, (500, n)); yy = y[i]
        def ba(x):
            xx = x[i]; pos = yy.sum(1); neg = (~yy).sum(1)
            return 0.5 * ((xx & yy).sum(1) / np.maximum(pos, 1) + (~xx & ~yy).sum(1) / np.maximum(neg, 1))
        out.append(ba(b) - ba(a))
    d = np.concatenate(out)
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

KIND = {"fp32": "FP32", "ptq": "PTQ", "qat": "QAT"}
rows, deltas, tests, flips, margins = [], [], [], [], []
for n, r, s in avail:
    P32 = preds(pid(n, r, s, "fp32"))[0]
    for k in ("fp32", "ptq", "qat"):
        if not C.have_preds(P, pid(n, r, s, k)):
            continue
        Pk = preds(pid(n, r, s, k))[0]
        for d in C.EVALSETS:
            m = C.metrics(Y[d], Pk[d]); rows.append(dict(arch=n, regime=r, seed=s, kind=KIND[k], dataset=d, **m))
            if k == "fp32":
                continue
            m0 = C.metrics(Y[d], P32[d])
            deltas.append(dict(arch=n, regime=r, seed=s, kind=KIND[k], dataset=d,
                               d_bacc=m["bacc"] - m0["bacc"], d_recall=m["recall"] - m0["recall"],
                               d_spec=m["specificity"] - m0["specificity"], d_auc=m["auc"] - m0["auc"]))
            fl = C.flips(Y[d], P32[d], Pk[d]); flips.append(dict(arch=n, regime=r, seed=s, kind=KIND[k], dataset=d, n=len(Y[d]), **fl))
            z = np.log(np.clip(P32[d], 1e-6, 1 - 1e-6) / np.clip(1 - P32[d], 1e-6, 1))
            flipped = (P32[d] > .5) != (Pk[d] > .5)
            for lo, hi in ((0, .5), (.5, 1), (1, 2), (2, 4), (4, np.inf)):
                mm = (np.abs(z) >= lo) & (np.abs(z) < hi)
                margins.append(dict(arch=n, regime=r, seed=s, kind=KIND[k], dataset=d, margin_bin=f"[{lo},{hi})",
                                    n=int(mm.sum()), flip_rate=float(flipped[mm].mean()) if mm.any() else np.nan))
            if s == median_seed(n, r):
                pv, n01, n10 = C.mcnemar_p(Y[d], P32[d], Pk[d]); lo, hi = bacc_boot(Y[d], P32[d], Pk[d])
                tests.append(dict(arch=n, regime=r, seed=s, kind=KIND[k], dataset=d, d_bacc=m["bacc"] - m0["bacc"],
                                  ci_lo=lo, ci_hi=hi, fp32_only_correct=n01, int8_only_correct=n10, p=pv))
S = pd.DataFrame(rows); D = pd.DataFrame(deltas); T = pd.DataFrame(tests); F = pd.DataFrame(flips); MG = pd.DataFrame(margins)
if len(T):
    T["p_holm"] = C.holm(T["p"].values)
S.to_csv(os.path.join(A, "student_runs.csv"), index=False); D.to_csv(os.path.join(A, "int8_deltas.csv"), index=False)
T.to_csv(os.path.join(A, "mcnemar_int8.csv"), index=False); F.to_csv(os.path.join(A, "flips.csv"), index=False)
MG.to_csv(os.path.join(A, "margins.csv"), index=False)

# regime effects (FP32 and INT8) against CE-std, paired by seed; McNemar on the median seeds
RE = []
for n in NAME:
    for r in [x for x in regimes_of(n) if x != "CE-std"]:
        for k in ("fp32", "ptq", "qat"):
            for d in C.EVALSETS:
                a = S[(S.arch == n) & (S.regime == "CE-std") & (S.kind == KIND[k]) & (S.dataset == d)].set_index("seed").bacc
                b = S[(S.arch == n) & (S.regime == r) & (S.kind == KIND[k]) & (S.dataset == d)].set_index("seed").bacc
                dd = (b - a).dropna()
                if dd.empty:
                    continue
                sa, sb = median_seed(n, "CE-std"), median_seed(n, r); pv = np.nan
                if C.have_preds(P, pid(n, "CE-std", sa, k)) and C.have_preds(P, pid(n, r, sb, k)):
                    pv = C.mcnemar_p(Y[d], preds(pid(n, "CE-std", sa, k))[0][d], preds(pid(n, r, sb, k))[0][d])[0]
                RE.append(dict(arch=n, regime=r, kind=KIND[k], dataset=d, n_seeds=len(dd),
                               d_bacc_mean=dd.mean(), d_bacc_sd=dd.std(ddof=1) if len(dd) > 1 else np.nan, p_median_seed=pv))
RE = pd.DataFrame(RE)
if len(RE):
    RE["p_holm"] = C.holm(RE["p_median_seed"].fillna(1).values)
RE.to_csv(os.path.join(A, "regime_effects.csv"), index=False)

# ReLU vs ReLU6: INT8 cost per seed, paired by seed (same seeds, same data)
R6 = []
n0, n6 = C.student_name(*C.MAIN_STUDENT), C.student_name(*C.RELU6_STUDENT)
for r in ("CE-std", "CAD"):
    for k in ("PTQ", "QAT"):
        for d in C.EVALSETS:
            a = D[(D.arch == n0) & (D.regime == r) & (D.kind == k) & (D.dataset == d)].set_index("seed").d_bacc
            b = D[(D.arch == n6) & (D.regime == r) & (D.kind == k) & (D.dataset == d)].set_index("seed").d_bacc
            j = a.index.intersection(b.index)
            if len(j) < 2:
                continue
            diff = b[j] - a[j]
            R6.append(dict(regime=r, kind=k, dataset=d, n_seeds=len(j), cost_relu=a[j].mean(), cost_relu_sd=a[j].std(ddof=1),
                           cost_relu6=b[j].mean(), cost_relu6_sd=b[j].std(ddof=1), diff_mean=diff.mean(),
                           diff_sd=diff.std(ddof=1), p_paired_t=st.ttest_rel(b[j], a[j]).pvalue))
R6 = pd.DataFrame(R6); R6.to_csv(os.path.join(A, "relu6_test.csv"), index=False)

# teacher: FP32 (5 seeds, all images), ladder on the common subset, weight-only sweep
TR = []
for s in C.CFG["SEEDS"]:
    if C.have_preds(P, f"T_fp32_s{s}"):
        pr = preds(f"T_fp32_s{s}")[0]
        for d in C.EVALSETS:
            TR.append(dict(model="MWAMNet", seed=s, kind="FP32", dataset=d, **C.metrics(Y[d], pr[d])))
TR = pd.DataFrame(TR); TR.to_csv(os.path.join(A, "teacher_fp32.csv"), index=False)
lad = {}
for f in sorted(os.listdir(P["preds"])):
    if f.startswith("T_") and f.endswith(".npz") and not f.startswith("T_w") and "ens" not in f:
        pr, meta, ix = preds(f[:-4]); lad[f[:-4]] = (pr, meta, ix)
ladder_seeds = sorted({m["seed"] for _, m, _ in lad.values() if m["quant"] in ("int8", "int8c")})
lad = {k: v for k, v in lad.items() if v[1]["seed"] in ladder_seeds}
common = {d: C.common_index([ix.get(d) for _, _, ix in lad.values()], len(Y[d])) for d in C.EVALSETS}
TL = []
for mid, (pr, meta, ix) in lad.items():
    ref = lad.get(f"T_fp32_s{meta['seed']}")
    for d in C.EVALSETS:
        y = Y[d][common[d]]; p = C.preds_on(pr[d], ix.get(d), common[d]); row = dict(quant=meta["quant"], seed=meta["seed"], dataset=d, **C.metrics(y, p))
        if ref is not None and meta["quant"] != "fp32":
            p0 = C.preds_on(ref[0][d], ref[2].get(d), common[d]); row.update({"flip_" + k: v for k, v in C.flips(y, p0, p).items()})
            row["p_mcnemar"] = C.mcnemar_p(y, p0, p)[0]
        TL.append(row)
TL = pd.DataFrame(TL); TL.to_csv(os.path.join(A, "teacher_ladder.csv"), index=False)
print("Teacher ladder images per dataset:", {d: len(v) for d, v in common.items()}, "| seeds:", ladder_seeds)

# weight-only bit-width sweeps (teacher: Notebook 01; student: Notebook 03)
WB = M[M.quant.astype(str).str.fullmatch(r"w\d") | (M.quant == "fp32")]
WB = WB[(WB.family == "teacher") | (WB.arch == C.student_name(*C.MAIN_STUDENT))]
WB = WB[WB.seed.astype(str) != "ens5"]
WB.to_csv(os.path.join(A, "bitwidth_runs.csv"), index=False)

def ms(x):
    return "%.1f ± %.1f" % (100 * x.mean(), 100 * x.std(ddof=1) if len(x) > 1 else 0)
print("\nINT8 cost (points of balanced accuracy, INT8 − FP32, mean ± SD over seeds; reference kernels):")
print(D.groupby(["arch", "regime", "kind", "dataset"]).d_bacc.apply(ms).unstack("dataset").to_string())
print("\nReLU vs ReLU6 (INT8 cost, points):")
print(R6.round(4).to_string(index=False) if len(R6) else "not available")

Teacher ladder images per dataset: {'din': 380, 'wild': 1000, 'dfire': 1000} | seeds: [42, 43, 46]

INT8 cost (points of balanced accuracy, INT8 − FP32, mean ± SD over seeds; reference kernels):
dataset                           dfire         din         wild
arch          regime  kind                                      
MNv1-0.25-96  CAD     PTQ     1.5 ± 0.7   0.1 ± 0.2   -0.0 ± 0.9
                      QAT    -0.1 ± 0.3   0.0 ± 0.0   -0.3 ± 0.1
              CE-std  PTQ     0.2 ± 0.5   0.1 ± 0.3   -0.0 ± 0.7
                      QAT    -0.3 ± 0.8   0.3 ± 0.3   -0.0 ± 0.4
MNv2-0.35-128 CAD     PTQ    -1.0 ± 2.0  -0.1 ± 0.2   -0.7 ± 1.5
                      QAT   -12.3 ± 2.1  -4.7 ± 1.8  -11.5 ± 0.6
              CE-std  PTQ    -0.9 ± 0.5   0.0 ± 0.0    0.8 ± 0.4
                      QAT    -8.9 ± 1.4  -3.3 ± 0.9  -10.1 ± 1.0
MNv3S-1-128   CAD     PTQ    -4.6 ± 2.1  -0.8 ± 0.3   -5.3 ± 1.6
                      QAT    -0.6 ± 1.7  -0.2 ± 0.2   -0.5 ± 0.5
              CE-conf PTQ

In [ ]:
#@title 8 · XAI stability: occlusion maps (FP32 vs INT8 student; teacher vs student)
G, PS, FILL = 7, 32, 128                         # 7 x 7 grid of 32-px patches on the 224-px image
rng = np.random.default_rng(7); pick = []
for d in C.EVALSETS:
    for c in (1, 0):
        idx = np.where(Y[d] == c)[0]; pick += [(d, int(i)) for i in rng.choice(idx, min(10, len(idx)), replace=False)]
Xs = np.stack([X224[d][i] for d, i in pick])

def occluded(x):
    out = [x]
    for gy in range(G):
        for gx in range(G):
            o = x.copy(); o[gy * PS:(gy + 1) * PS, gx * PS:(gx + 1) * PS] = FILL; out.append(o)
    return np.stack(out)

def occ_maps(predict):
    maps = np.zeros((len(Xs), G, G), np.float32)
    for i, x in enumerate(Xs):
        p = predict(occluded(x)); maps[i] = (p[0] - p[1:]).reshape(G, G)
    return maps

f_maps = os.path.join(A, "xai_maps.npz")
if os.path.exists(f_maps):
    MAPS = dict(np.load(f_maps))
else:
    MAPS = {}
    med = C.load_json(os.path.join(P["data"], "teacher_median_seed.json"))["median_seed"]
    T = C.build_teacher_e2e(os.path.join(P["models"], f"teacher_head_s{med}.weights.h5"))
    MAPS["teacher_fp32"] = occ_maps(lambda X: T.predict(X.astype("float32"), batch_size=25, verbose=0).ravel())
    del T; keras.backend.clear_session(); gc.collect()
    for n in (C.student_name(*C.MAIN_STUDENT), C.student_name(*C.RELU6_STUDENT)):
        arch, alpha, res = NAME[n]; s = median_seed(n, "CAD")
        if s is None:
            continue
        core = C.build_student(arch, alpha, res, weights=None)
        core.load_weights(os.path.join(P["models"], f"S_{n}_CAD_s{s}_fp32.weights.h5"))
        rs = lambda X: C.resize_u8(X, res)
        MAPS[f"{n}_fp32"] = occ_maps(lambda X: 1 / (1 + np.exp(-core(C.to_unit(rs(X)), training=False).numpy().ravel())))
        for k in ("ptq", "qat"):
            f = tfl(n, "CAD", s, k)
            if os.path.exists(f):
                b = open(f, "rb").read()
                MAPS[f"{n}_{k}"] = occ_maps(lambda X: C.tflite_predict(b, rs(X), bs=50, kernels="ref"))
        keras.backend.clear_session()
    np.savez_compressed(f_maps, **MAPS, pick_dataset=np.array([d for d, _ in pick]), pick_index=np.array([i for _, i in pick]))

def top_iou(a, b, k=5):
    ta, tb = set(np.argsort(-a.ravel())[:k]), set(np.argsort(-b.ravel())[:k])
    return len(ta & tb) / len(ta | tb)
XR = []
n0 = C.student_name(*C.MAIN_STUDENT); n6 = C.student_name(*C.RELU6_STUDENT)
pairs = [(f"{n0}_fp32", f"{n0}_ptq"), (f"{n0}_fp32", f"{n0}_qat"), (f"{n6}_fp32", f"{n6}_ptq"),
         (f"{n6}_fp32", f"{n6}_qat"), ("teacher_fp32", f"{n0}_fp32")]
for a, b in pairs:
    if a in MAPS and b in MAPS:
        for i, (d, idx) in enumerate(pick):
            rho = st.spearmanr(MAPS[a][i].ravel(), MAPS[b][i].ravel()).correlation
            XR.append(dict(pair=f"{a} vs {b}", dataset=d, label=int(Y[d][idx]), image=idx,
                           spearman=rho, top5_iou=top_iou(MAPS[a][i], MAPS[b][i])))
XR = pd.DataFrame(XR); XR.to_csv(os.path.join(A, "xai_stability.csv"), index=False)
if len(XR):
    print(XR.groupby(["pair", "dataset"])[["spearman", "top5_iou"]].median().round(3).to_string())

12683000/12683000 [==============================] - 0s 0us/step


/tmp/ipykernel_1559/306456235.py:57: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = st.spearmanr(MAPS[a][i].ravel(), MAPS[b][i].ravel()).correlation


                                               spearman  top5_iou
pair                                  dataset                    
MNv3S-1-128_fp32 vs MNv3S-1-128_ptq   dfire       0.657     0.111
                                      din         0.671     0.056
                                      wild        0.689     0.250
MNv3S-1-128_fp32 vs MNv3S-1-128_qat   dfire       0.722     0.429
                                      din         0.643     0.000
                                      wild        0.759     0.429
MNv3S6-1-128_fp32 vs MNv3S6-1-128_ptq dfire       0.851     0.548
                                      din         0.718     0.111
                                      wild        0.670     0.181
MNv3S6-1-128_fp32 vs MNv3S6-1-128_qat dfire       0.870     0.429
                                      din         0.601     0.111
                                      wild        0.708     0.250
teacher_fp32 vs MNv3S-1-128_fp32      dfire       0.164     0.000
          

In [ ]:
#@title 9 · Summary
print("Files in results/analysis/:")
for f in sorted(os.listdir(A)):
    print("  ", f)

Files in results/analysis/:
   bitwidth_runs.csv
   flips.csv
   int8_deltas.csv
   margins.csv
   mcnemar_int8.csv
   overflow
   overflow_summary.csv
   regime_effects.csv
   relu6_test.csv
   runtime_agreement.csv
   student_runs.csv
   teacher_fp32.csv
   teacher_ladder.csv
   xai_maps.npz
   xai_stability.csv
